In [1]:
import pandas as pandas
import sqlite3
import os
import sys
import platform
import pandas as pd

In [2]:
#For each example in the blimp dataset, find the length and save 
#For examples with the two sentences having different lengths, decide what to do (easier is first to just save both the lengths separately, can always be combined later)

if "pop-os" in platform.node():
    ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/"
else:
    ROOT = r'/gpfs/home4/athamma/repo/ss-llm/nanoGPT/'#Given a set of details, return a dataframe compiled with the details

RESULTS_ROOT = os.path.join(ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)


#Other Utils
import io
import sys
import contextlib

@contextlib.contextmanager
def silence_prints():
    sys.stdout, old = io.StringIO(), sys.stdout
    try:
        yield
    finally:
        sys.stdout = old


In [3]:
#List of Model IDs that have Masking of exponential_new and have decay rate 2 and Echoic Memory 10
#Alternatively all models with exoponential_new and decay rate 2 

MODEL_QUERY = """    SELECT DISTINCT InvertedMaskModelSurprisalScores.ModelID, Model.OutputFolderName, Model.BatchSize, Model.Dataset, Model.Seed, Model.MaskType FROM InvertedMaskModelSurprisalScores
JOIN Model on Model.ModelID = InvertedMaskModelSurprisalScores.ModelID
WHERE Model.NumLayers = 6 
AND Model.BatchSize = 32
AND ((Model.EchoicMemory=10 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2) OR (Model.MaskType="Non" AND Model.CurriculumLearning=False)) 
AND Model.Dataset in ("babylm_full_bpe_8k", "babylm_full_bpe_100M_8k")  
AND Model.ModelID not in (5496427, 8456913)
ORDER BY Seed, BatchSize, Dataset, MaskType

"""

model_list_df = pd.read_sql_query(MODEL_QUERY, conn)
model_list_df


,ModelID,OutputFolderName,BatchSize,Dataset,Seed,MaskType
0,8465085,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,32,babylm_full_bpe_100M_8k,9,exponential_new
1,6839425,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-683...,32,babylm_full_bpe_8k,9,exponential_new
2,8465082,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,32,babylm_full_bpe_100M_8k,42,exponential_new
3,6839403,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-683...,32,babylm_full_bpe_8k,42,exponential_new
4,8465091,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,32,babylm_full_bpe_100M_8k,466,exponential_new
5,6839430,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-683...,32,babylm_full_bpe_8k,466,exponential_new
6,8465086,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,32,babylm_full_bpe_100M_8k,616,exponential_new
7,6839426,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-683...,32,babylm_full_bpe_8k,616,exponential_new
8,8465090,out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em1...,32,babylm_full_bpe_100M_8k,869,exponential_new
9,6839429,out-babylm_full_bpe_8k-6x6-mask_ee002_em10-683...,32,babylm_full_bpe_8k,869,exponential_new


In [4]:
#Query to get Average Surprisal Scores per corpus for models in the model_list_df from ModelSurprisalScores and InvertedMaskModelSurprisalScores and join on Model

AGGREGATED_MODEL_SURPRISAL_QUERY = """
    WITH ModelScoreSubset AS (
        SELECT ModelSurprisalScores.ModelID, ModelSurprisalScores.StoryWordID, ModelSurprisalScores.SurprisalScore, Story.CorpusID, Story.StoryID
        FROM ModelSurprisalScores 
        JOIN Story ON ModelSurprisalScores.StoryWordID = Story.StoryWordID
        WHERE ModelSurprisalScores.ModelID IN ({})),

        InvertedModelScoreSubset AS (
            SELECT InvertedMaskModelSurprisalScores.ModelID, InvertedMaskModelSurprisalScores.StoryWordID, InvertedMaskModelSurprisalScores.SurprisalScore, Story.CorpusID, Story.StoryID
            FROM InvertedMaskModelSurprisalScores
            JOIN Story ON InvertedMaskModelSurprisalScores.StoryWordID = Story.StoryWordID
            WHERE InvertedMaskModelSurprisalScores.ModelID IN ({})
            ),

        ModelScoreAggregated AS (
            SELECT ModelID, CorpusID, AVG(SurprisalScore) AS AvgSurprisalScore
            FROM ModelScoreSubset
            GROUP BY ModelID, CorpusID
        ),
        InvertedModelScoreAggregated AS (
            SELECT ModelID, CorpusID, AVG(SurprisalScore) AS AvgSurprisalScore
            FROM InvertedModelScoreSubset
            GROUP BY ModelID, CorpusID
        ),

        JoinedModelScoreAggregated AS (
            SELECT ModelScoreAggregated.ModelID, ModelScoreAggregated.CorpusID, ModelScoreAggregated.AvgSurprisalScore AS AvgSurprisalScore, InvertedModelScoreAggregated.AvgSurprisalScore AS InvertedAvgSurprisalScore
            FROM ModelScoreAggregated
            JOIN InvertedModelScoreAggregated ON ModelScoreAggregated.ModelID = InvertedModelScoreAggregated.ModelID AND ModelScoreAggregated.CorpusID = InvertedModelScoreAggregated.CorpusID
        )
        SELECT JoinedModelScoreAggregated.ModelID, JoinedModelScoreAggregated.CorpusID, JoinedModelScoreAggregated.AvgSurprisalScore, JoinedModelScoreAggregated.InvertedAvgSurprisalScore, Model.Seed, Model.Dataset
        FROM JoinedModelScoreAggregated
        JOIN Model ON JoinedModelScoreAggregated.ModelID = Model.ModelID
        ORDER BY JoinedModelScoreAggregated.CorpusID DESC, Model.Seed, Model.Dataset
        

"""

model_aggregated_surprisal_df = pd.read_sql(AGGREGATED_MODEL_SURPRISAL_QUERY.format(
    ', '.join(map(str, model_list_df["ModelID"].unique())),
    ', '.join(map(str, model_list_df["ModelID"].unique())),
    ', '.join(map(str, model_list_df["ModelID"].unique()))
), conn)
model_aggregated_surprisal_df


,ModelID,CorpusID,AvgSurprisalScore,InvertedAvgSurprisalScore,Seed,Dataset
0,8465085,2,5.779832,8.754942,9,babylm_full_bpe_100M_8k
1,6839425,2,6.485959,8.371690,9,babylm_full_bpe_8k
2,8465082,2,5.792760,9.417142,42,babylm_full_bpe_100M_8k
3,6839403,2,6.493478,8.989988,42,babylm_full_bpe_8k
4,8465091,2,5.786320,8.262520,466,babylm_full_bpe_100M_8k
5,6839430,2,6.477534,8.463284,466,babylm_full_bpe_8k
6,8465086,2,5.783335,8.416255,616,babylm_full_bpe_100M_8k
7,6839426,2,6.514423,8.844314,616,babylm_full_bpe_8k
8,8465090,2,5.785675,8.569172,869,babylm_full_bpe_100M_8k
9,6839429,2,6.474080,8.837321,869,babylm_full_bpe_8k


In [5]:
model_aggregated_surprisal_df["diff"] = model_aggregated_surprisal_df["AvgSurprisalScore"] - model_aggregated_surprisal_df["InvertedAvgSurprisalScore"]

model_aggregated_surprisal_df

,ModelID,CorpusID,AvgSurprisalScore,InvertedAvgSurprisalScore,Seed,Dataset,diff
0,8465085,2,5.779832,8.754942,9,babylm_full_bpe_100M_8k,-2.975110
1,6839425,2,6.485959,8.371690,9,babylm_full_bpe_8k,-1.885731
2,8465082,2,5.792760,9.417142,42,babylm_full_bpe_100M_8k,-3.624382
3,6839403,2,6.493478,8.989988,42,babylm_full_bpe_8k,-2.496510
4,8465091,2,5.786320,8.262520,466,babylm_full_bpe_100M_8k,-2.476200
5,6839430,2,6.477534,8.463284,466,babylm_full_bpe_8k,-1.985749
6,8465086,2,5.783335,8.416255,616,babylm_full_bpe_100M_8k,-2.632920
7,6839426,2,6.514423,8.844314,616,babylm_full_bpe_8k,-2.329890
8,8465090,2,5.785675,8.569172,869,babylm_full_bpe_100M_8k,-2.783497
9,6839429,2,6.474080,8.837321,869,babylm_full_bpe_8k,-2.363241


In [42]:
#List of Model IDs that have Masking of exponential_new and have decay rate 2 and Echoic Memory 10
#Alternatively all models with exoponential_new and decay rate 2 

PLAIN_MODEL_QUERY = """    SELECT DISTINCT ModelSurprisalScores.ModelID, Model.BatchSize, Model.Dataset, Model.Seed, Model.MaskType FROM ModelSurprisalScores
JOIN Model on Model.ModelID = ModelSurprisalScores.ModelID
WHERE Model.NumLayers = 6 
AND Model.BatchSize = 32
AND ((Model.EchoicMemory=10 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2) OR (Model.MaskType="Non" AND Model.CurriculumLearning=False)) 
AND Model.Dataset in ("babylm_full_bpe_8k", "babylm_full_bpe_100M_8k")  
AND Model.ModelID not in (5496427, 8456913)
ORDER BY Seed, BatchSize, Dataset, MaskType

"""

plain_model_list_df = pd.read_sql_query(PLAIN_MODEL_QUERY, conn)

plain_model_list_df = plain_model_list_df.pivot(index=["BatchSize", "Dataset", "Seed"], columns=["MaskType"], values=["ModelID"]).reset_index()

plain_model_list_df.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in plain_model_list_df.columns]

plain_model_list_df


,BatchSize,Dataset,Seed,ModelID_Non,ModelID_exponential_new
0,32,babylm_full_bpe_100M_8k,9,8465733,8465085
1,32,babylm_full_bpe_100M_8k,42,8465604,8465082
2,32,babylm_full_bpe_100M_8k,466,8465611,8465091
3,32,babylm_full_bpe_100M_8k,616,8465607,8465086
4,32,babylm_full_bpe_100M_8k,869,8465610,8465090
5,32,babylm_full_bpe_100M_8k,1337,8096895,8465077
6,32,babylm_full_bpe_100M_8k,2347,8465605,8465084
7,32,babylm_full_bpe_100M_8k,6747,8465609,8465089
8,32,babylm_full_bpe_100M_8k,11111,8465612,8465093
9,32,babylm_full_bpe_100M_8k,46674,8465608,8465087


In [43]:
PLAIN_DIRECT_AGGREGATED_MODEL_SURPRISAL_QUERY = """
    WITH ModelScoreSubset AS (
        SELECT ModelSurprisalScores.ModelID, ModelSurprisalScores.StoryWordID, ModelSurprisalScores.SurprisalScore, Story.CorpusID, Story.StoryID
        FROM ModelSurprisalScores 
        JOIN Story ON ModelSurprisalScores.StoryWordID = Story.StoryWordID
        WHERE ModelSurprisalScores.ModelID IN ({})),

        ModelScoreAggregated AS (
            SELECT ModelID, CorpusID, AVG(SurprisalScore) AS AvgSurprisalScore
            FROM ModelScoreSubset
            GROUP BY ModelID, CorpusID
        )
        SELECT ModelScoreAggregated.ModelID, ModelScoreAggregated.CorpusID, ModelScoreAggregated.AvgSurprisalScore, Model.Seed, Model.Dataset
        FROM ModelScoreAggregated
        JOIN Model ON ModelScoreAggregated.ModelID = Model.ModelID
        ORDER BY ModelScoreAggregated.CorpusID DESC, Model.Seed, Model.Dataset
        

"""
plain_model_aggregated_surprisal_df = pd.read_sql(PLAIN_DIRECT_AGGREGATED_MODEL_SURPRISAL_QUERY.format(
    ', '.join(map(str, plain_model_list_df["ModelID_Non"].unique()))), conn)
plain_model_aggregated_surprisal_df

,ModelID,CorpusID,AvgSurprisalScore,Seed,Dataset
0,8465733,2,5.805582,9,babylm_full_bpe_100M_8k
1,6892214,2,6.490653,9,babylm_full_bpe_8k
2,8465604,2,5.822579,42,babylm_full_bpe_100M_8k
3,6892212,2,6.484804,42,babylm_full_bpe_8k
4,8465611,2,5.814573,466,babylm_full_bpe_100M_8k
5,6892220,2,6.487708,466,babylm_full_bpe_8k
6,8465607,2,5.814417,616,babylm_full_bpe_100M_8k
7,6892216,2,6.473027,616,babylm_full_bpe_8k
8,8465610,2,5.793157,869,babylm_full_bpe_100M_8k
9,6892219,2,6.506540,869,babylm_full_bpe_8k


In [44]:
model_aggregated_surprisal_df = model_aggregated_surprisal_df.merge(plain_model_list_df[["ModelID_Non", "ModelID_exponential_new"]], left_on="ModelID", right_on="ModelID_exponential_new", how="left").merge(plain_model_aggregated_surprisal_df[["ModelID", "CorpusID","AvgSurprisalScore"]], left_on=["ModelID_Non","CorpusID"], right_on=["ModelID","CorpusID"], how="left", suffixes=("", "_plain")).drop(columns=["ModelID_exponential_new", "ModelID_Non"])

model_aggregated_surprisal_df 

,ModelID,CorpusID,AvgSurprisalScore,InvertedAvgSurprisalScore,Seed,Dataset,diff,ModelID_plain,AvgSurprisalScore_plain
0,8465085,2,5.779832,8.754942,9,babylm_full_bpe_100M_8k,-2.975110,8465733,5.805582
1,6839425,2,6.485959,8.371690,9,babylm_full_bpe_8k,-1.885731,6892214,6.490653
2,8465082,2,5.792760,9.417142,42,babylm_full_bpe_100M_8k,-3.624382,8465604,5.822579
3,6839403,2,6.493478,8.989988,42,babylm_full_bpe_8k,-2.496510,6892212,6.484804
4,8465091,2,5.786320,8.262520,466,babylm_full_bpe_100M_8k,-2.476200,8465611,5.814573
5,6839430,2,6.477534,8.463284,466,babylm_full_bpe_8k,-1.985749,6892220,6.487708
6,8465086,2,5.783335,8.416255,616,babylm_full_bpe_100M_8k,-2.632920,8465607,5.814417
7,6839426,2,6.514423,8.844314,616,babylm_full_bpe_8k,-2.329890,6892216,6.473027
8,8465090,2,5.785675,8.569172,869,babylm_full_bpe_100M_8k,-2.783497,8465610,5.793157
9,6839429,2,6.474080,8.837321,869,babylm_full_bpe_8k,-2.363241,6892219,6.506540


In [ ]:
#Use the Masked Models from the inverted folder but use non mask from normal data csv and use that to plot Fig 5